In [ ]:
# --- Celda 1: preparación ---
import sys, os, numpy as np, matplotlib.pyplot as plt
raiz = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, raiz)
from src.barrido import correr, resumen, barrido_2d, tornado
from src.euler_model import load_params

nombres = ["Sensibles", "Tolerantes", "Resistentes"]
pB, _ = load_params("B")
pC, _ = load_params("C")

In [ ]:
# --- Celda 2: escenario B frente a C (100 días) ---
tB, XB = correr(pB, dias=100)
tC, XC = correr(pC, dias=100)
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
casos = [(tB, XB, f"B  (µST={pB['muST']:.0e}, µTR={pB['muTR']:.0e})"),
         (tC, XC, f"C  (µST={pC['muST']:.0e}, µTR={pC['muTR']:.0e})")]
for a, (t, X, tit) in zip(ax, casos):
    for i, n in enumerate(nombres):
        a.semilogy(t, np.maximum(X[:, i], 0.1), label=n)
    a.set(title=tit, xlabel="Tiempo (días)"); a.grid(alpha=0.3)
ax[0].set_ylabel("Densidad (UFC/mL)"); ax[0].legend()
plt.tight_layout(); plt.show()
for nom, (t, X) in {"B": (tB, XB), "C": (tC, XC)}.items():
    print(nom, {k: round(v, 3) for k, v in resumen(t, X).items()})

In [ ]:
# --- Celda 3: barrido 2D de µST x µTR (tarda ~30 s en total) ---
mst = np.logspace(-6, -2, 7); mtr = np.logspace(-7, -3, 7)

def barrido_con_R0(R0):
    q = dict(pB); q["x0"] = dict(pB["x0"]); q["x0"]["R"] = R0
    return barrido_2d(q, mst, mtr, dias=150)

res10, res0 = barrido_con_R0(10.0), barrido_con_R0(0.0)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, M, tit in [(ax[0], res10["t_dom_R"], "R₀ = 10 (resistentes preexistentes)"),
                  (ax[1], res0["t_dom_R"], "R₀ = 0 (resistencia solo por mutación)")]:
    im = a.pcolormesh(np.log10(mst), np.log10(mtr), M, shading="nearest", cmap="viridis")
    a.set(xlabel="log10 µST (S→T)", ylabel="log10 µTR (T→R)", title=tit)
    a.plot(np.log10(pB["muST"]), np.log10(pB["muTR"]), "wo", mec="k", label="B")
    a.plot(np.log10(pC["muST"]), np.log10(pC["muTR"]), "w^", mec="k", label="C")
    plt.colorbar(im, ax=a, label="días hasta dominancia de R")
ax[0].legend()
plt.tight_layout(); plt.show()

In [ ]:
# --- Celda 4: sensibilidad local (tornado) ---
tor = tornado(pB)
orden = sorted(tor, key=lambda n: abs(np.nan_to_num(tor[n][2] - tor[n][0])))
base = tor["rR"][1]
fig, ax = plt.subplots(figsize=(7, 4))
for k, n in enumerate(orden):
    tm, _, tp, el = tor[n]
    ax.barh(k, tm - base, left=base, color="tab:blue", label="-20 %" if k == 0 else None)
    ax.barh(k, tp - base, left=base, color="tab:orange", label="+20 %" if k == 0 else None)
ax.set_yticks(range(len(orden))); ax.set_yticklabels(orden)
ax.axvline(base, color="k", lw=1)
ax.set(xlabel="días hasta dominancia de R", title=f"Sensibilidad local (base = {base:.1f} d)")
ax.legend(); plt.tight_layout(); plt.show()
print("Elasticidades:", {n: round(tor[n][3], 2) for n in orden[::-1]})